In [0]:
%run ./01_config

In [0]:
"""
04_bronze_ingest.py  —  Bronze layer: raw ingest

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 12.
"""

# 04 — Bronze ingest

# Reads the seven SAP PM entity extracts plus the ground-truth disclosure file from the
# volume and lands them as Delta tables as-is: every column stays a string, nothing is
# cleaned, nothing is dropped. Typing and conforming happen in silver (notebook 05).

# Reading all-string is deliberate — it means v2 and v2.1 extracts ingest through the same
# code even though v2.1 adds columns (frailty, QMNUM_ORIG, criticality). Schema drift is
# then a silver-layer concern, reported rather than crashing the ingest.

# Shared configuration from '01_config' is assumed to be in scope.

from pyspark.sql import functions as F

use_project_schema()

def ingest(entity: str, source_path: str, target_suffix: str = None) -> dict:
    """Land one CSV as a bronze Delta table with provenance columns."""
    df = (
        spark.read
             .option("header", True)
             .option("inferSchema", False)
             .option("nullValue", "")
             .csv(source_path)
    )
    df = (df.withColumn("_source_file", F.lit(source_path))
            .withColumn("_dataset_version", F.lit(DATASET_VERSION))
            .withColumn("_ingest_ts", F.current_timestamp()))

    target = tbl(f"bronze_{(target_suffix or entity).lower()}")
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(target))

    n = spark.table(target).count()
    return {"entity": entity, "table": target, "rows": n,
            "columns": [c for c in df.columns if not c.startswith("_")]}

landed = [f.name for f in dbutils.fs.ls(LANDING) if f.name.lower().endswith(".csv")]
print(f"CSV files found in {LANDING}:")
for f in landed:
    print("  ", f)

missing = [e for e in CONTRACT_ENTITIES if f"{e}.csv" not in landed]
if missing:
    raise FileNotFoundError(
        f"Missing entity extracts: {missing}. Run the generator with "
        f"OUTPUT_DIR = '{LANDING}' before running this notebook."
    )

results = [ingest(e, f"{LANDING}/{e}.csv") for e in CONTRACT_ENTITIES]

for r in results:
    print(f"{r['entity']:6s} -> {r['rows']:>8,} rows | {len(r['columns'])} cols")

print(f"\nTotal transactional + master records: {sum(r['rows'] for r in results):,}")

# Ground-truth parameters

# Kept in a separate folder and a separate table. Nothing in the silver or gold layers
# joins to it — the two-level validation design (Section 4.1.1) depends on the fitted models
# never seeing the generating parameters. It is loaded here only so the parameter-recovery
# comparison in Section 4.2.3 can be run from the same catalog.

gt_files = [f.name for f in dbutils.fs.ls(GROUND_TRUTH) if f.name.lower().endswith(".csv")]

if "simulation_parameters.csv" in gt_files:
    r = ingest("simulation_parameters", f"{GROUND_TRUTH}/simulation_parameters.csv", "ground_truth_params")
    print(f"ground truth -> {r['rows']} classes")
    display(spark.table(tbl("bronze_ground_truth_params")))
else:
    print(f"WARNING: simulation_parameters.csv not found in {GROUND_TRUTH}. "
          "Parameter recovery (Test T3) cannot run until it is placed there.")

# Run manifest

# One row per ingest run, appended. This is the audit trail that lets any reported metric
# be traced back to a specific dataset version and load (NFR1).

manifest = spark.createDataFrame(
    [(DATASET_VERSION, r["entity"], r["table"], r["rows"], ",".join(r["columns"])) for r in results],
    "dataset_version string, entity string, table_name string, row_count long, columns string"
).withColumn("ingested_at", F.current_timestamp())

(manifest.write.mode("append")
         .option("mergeSchema", "true")
         .saveAsTable(tbl("_ingest_manifest")))

display(spark.table(tbl("_ingest_manifest")).orderBy(F.col("ingested_at").desc()))